In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from typing import List, Tuple
import numpy as np


%matplotlib widget

In [ ]:
FILE_PATH: str = "..\\..\\data\\carla\\scenario1_2_cars_speeding\\radar_points_world.csv"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from typing import List, Tuple
import numpy as np

class RadarDataLoader:
    def __init__(
        self,
        filePath: str,
        colX: str = "x_sensor",
        colY: str = "y_sensor",
        colZ: str = "z_sensor",
        colVelocity: str = "radial_velocity",
        colFrame: str = "frame",
        colTimestamp: str = "timestamp"
    ):
        self.filePath: str = filePath
        self.colX: str = colX
        self.colY: str = colY
        self.colZ: str = colZ
        self.colVelocity: str = colVelocity
        self.colFrame: str = colFrame
        self.colTimestamp: str = colTimestamp
        self.dataFrame: pd.DataFrame = pd.DataFrame()
        self.uniqueFrames: List[int] = []
        self.startTimestamp: float = 0.0
        self.totalDuration: float = 0.0

    def load(self) -> bool:
        try:
            self.dataFrame = pd.read_csv(self.filePath)
            self.uniqueFrames = sorted(self.dataFrame[self.colFrame].unique().tolist())
            self.startTimestamp = float(self.dataFrame[self.colTimestamp].min())
            self.totalDuration = float(self.dataFrame[self.colTimestamp].max()) - self.startTimestamp
            return not self.dataFrame.empty
        except Exception:
            return False

    def getFrameData(self, frameId: int) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series, float]:
        frameMask = self.dataFrame[self.colFrame] == frameId
        frameData = self.dataFrame[frameMask]
        currentTimestamp = float(frameData[self.colTimestamp].iloc[0])
        relativeTime = currentTimestamp - self.startTimestamp
        return (
            frameData[self.colX],
            frameData[self.colY],
            frameData[self.colZ],
            frameData[self.colVelocity],
            relativeTime
        )

    def getAxisLimits(self) -> List[Tuple[float, float]]:
        return [
            (float(self.dataFrame[self.colX].min()), float(self.dataFrame[self.colX].max())),
            (float(self.dataFrame[self.colY].min()), float(self.dataFrame[self.colY].max())),
            (float(self.dataFrame[self.colZ].min()), float(self.dataFrame[self.colZ].max()))
        ]

    def getVelocityLimits(self) -> Tuple[float, float]:
        return (
            float(self.dataFrame[self.colVelocity].min()),
            float(self.dataFrame[self.colVelocity].max())
        )

class RadarVisualizer:
    def __init__(
        self,
        dataLoader: RadarDataLoader,
        fps: int = 24,
        velocityThreshold: float = 0.3,
        persistStaticPoints: bool = False,
        removeStaticPoints: bool = True,
        repeatAnimation: bool = False
    ):
        self.dataLoader: RadarDataLoader = dataLoader
        self.fps: int = fps
        self.velocityThreshold: float = velocityThreshold
        self.persistStaticPoints: bool = persistStaticPoints
        self.removeStaticPoints: bool = removeStaticPoints
        self.repeatAnimation: bool = repeatAnimation
        self.animationInterval: int = 1000 // self.fps

        self.fig: plt.Figure = plt.figure(figsize=(12, 8), facecolor="#121212")
        self.ax = self.fig.add_subplot(111, projection="3d")
        self.ax.set_facecolor("#121212")
        
        self.scatter = self.ax.scatter([], [], [], c=[], cmap="coolwarm", s=15)
        self.timeText = self.ax.text2D(0.05, 0.95, "", transform=self.ax.transAxes, color="white", fontsize=10)

        self.staticX: List[float] = []
        self.staticY: List[float] = []
        self.staticZ: List[float] = []
        self.staticV: List[float] = []
        self.animation: FuncAnimation | None = None

    def initializeAxes(self) -> None:
        limits = self.dataLoader.getAxisLimits()
        vMin, vMax = self.dataLoader.getVelocityLimits()
        
        self.ax.set_xlim3d(limits[0])
        self.ax.set_ylim3d(limits[1])
        self.ax.set_zlim3d(limits[2])
        
        self.ax.set_xlabel("X (Sensor) [m]", color="white")
        self.ax.set_ylabel("Y (Sensor) [m]", color="white")
        self.ax.set_zlabel("Z (Sensor) [m]", color="white")
        self.ax.set_title("Radar 3D Velocity Map", color="white")
        
        self.ax.xaxis.label.set_color("white")
        self.ax.yaxis.label.set_color("white")
        self.ax.zaxis.label.set_color("white")
        self.ax.tick_params(axis="x", colors="white")
        self.ax.tick_params(axis="y", colors="white")
        self.ax.tick_params(axis="z", colors="white")
        
        self.ax.view_init(elev=20, azim=-35)
        self.scatter.set_clim(vMin, vMax)
        
        colorBar = self.fig.colorbar(self.scatter, ax=self.ax, pad=0.1)
        colorBar.set_label("Radial Velocity [m/s]", color="white")
        plt.setp(plt.getp(colorBar.ax.axes, "yticklabels"), color="white")

        self.ax.scatter([0], [0], [0], color="green", marker="X", s=100, label="Sensor Origin")
        self.ax.legend(loc="upper right")

    def update(self, frameId: int):
        x, y, z, velocity, relTime = self.dataLoader.getFrameData(frameId)
        isMoving = velocity.abs() >= self.velocityThreshold
        isStatic = velocity.abs() < self.velocityThreshold

        if self.removeStaticPoints:
            outX, outY, outZ, outV = x[isMoving].tolist(), y[isMoving].tolist(), z[isMoving].tolist(), velocity[isMoving].tolist()
        elif self.persistStaticPoints:
            self.staticX.extend(x[isStatic].tolist())
            self.staticY.extend(y[isStatic].tolist())
            self.staticZ.extend(z[isStatic].tolist())
            self.staticV.extend(velocity[isStatic].tolist())
            outX = self.staticX + x[isMoving].tolist()
            outY = self.staticY + y[isMoving].tolist()
            outZ = self.staticZ + z[isMoving].tolist()
            outV = self.staticV + velocity[isMoving].tolist()
        else:
            outX, outY, outZ, outV = x.tolist(), y.tolist(), z.tolist(), velocity.tolist()

        self.scatter._offsets3d = (outX, outY, outZ)
        self.scatter.set_array(np.array(outV))
        
        self.timeText.set_text(f"Time: {relTime:.2f}s / {self.dataLoader.totalDuration:.2f}s")
        return self.scatter, self.timeText

    def start(self) -> FuncAnimation:
        self.initializeAxes()
        self.animation = FuncAnimation(
            self.fig,
            self.update,
            frames=self.dataLoader.uniqueFrames,
            interval=self.animationInterval,
            blit=False,
            repeat=self.repeatAnimation
        )
        return self.animation



In [ ]:
if __name__ == "__main__":
    filePath: str = "..\\..\\data\\carla\\scenario1_2_cars_speeding\\radar_points_world.csv"
    loader = RadarDataLoader(filePath=filePath)
    if loader.load():
        visualizer = RadarVisualizer(
            dataLoader=loader,
            fps=24,
            velocityThreshold=0.3,
            persistStaticPoints=False,
            removeStaticPoints=True,
            repeatAnimation=False
        )
        radarAnim = visualizer.start()
        plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from typing import List, Tuple, Optional
import numpy as np
from IPython.display import Video

FILE_PATH = "data/carla/scenario5_empty_road/radar_points_world.csv"
OUTPUT_VIDEO_PATH = "radar_visualization.mp4"
FPS = 24
VELOCITY_THRESHOLD = 0.3
REMOVE_STATIC = True
PERSIST_STATIC = False
BITRATE = 2000
FIG_SIZE = (12, 8)
THEME_COLOR = "#121212"


class RadarVisualizer:
    def __init__(
        self,
        dataLoader: RadarDataLoader,
        fps: int = 24,
        velocityThreshold: float = 0.3,
        persistStaticPoints: bool = False,
        removeStaticPoints: bool = True,
        repeatAnimation: bool = False
    ):
        self.dataLoader: RadarDataLoader = dataLoader
        self.fps: int = fps
        self.velocityThreshold: float = velocityThreshold
        self.persistStaticPoints: bool = persistStaticPoints
        self.removeStaticPoints: bool = removeStaticPoints
        self.repeatAnimation: bool = repeatAnimation
        self.animationInterval: int = 1000 // self.fps

        self.fig: plt.Figure = plt.figure(figsize=FIG_SIZE, facecolor=THEME_COLOR)
        self.ax = self.fig.add_subplot(111, projection="3d")
        self.ax.set_facecolor(THEME_COLOR)
        
        self.scatter = self.ax.scatter([], [], [], c=[], cmap="coolwarm", s=15)
        self.timeText = self.ax.text2D(0.05, 0.95, "", transform=self.ax.transAxes, color="white", fontsize=10)

        self.staticX: List[float] = []
        self.staticY: List[float] = []
        self.staticZ: List[float] = []
        self.staticV: List[float] = []
        self.animation: Optional[FuncAnimation] = None

    def initializeAxes(self) -> None:
        limits = self.dataLoader.getAxisLimits()
        vMin, vMax = self.dataLoader.getVelocityLimits()
        
        self.ax.set_xlim3d(limits[0])
        self.ax.set_ylim3d(limits[1])
        self.ax.set_zlim3d(limits[2])
        
        self.ax.set_xlabel("X (Sensor) [m]", color="white")
        self.ax.set_ylabel("Y (Sensor) [m]", color="white")
        self.ax.set_zlabel("Z (Sensor) [m]", color="white")
        self.ax.set_title("Radar 3D Velocity Map", color="white")
        
        for axis in [self.ax.xaxis, self.ax.yaxis, self.ax.zaxis]:
            axis.label.set_color("white")
            self.ax.tick_params(axis=axis.axis_name, colors="white")
        
        self.ax.view_init(elev=20, azim=-35)
        self.scatter.set_clim(vMin, vMax)
        
        colorBar = self.fig.colorbar(self.scatter, ax=self.ax, pad=0.1)
        colorBar.set_label("Radial Velocity [m/s]", color="white")
        plt.setp(plt.getp(colorBar.ax.axes, "yticklabels"), color="white")

        self.ax.scatter([0], [0], [0], color="green", marker="X", s=100, label="Sensor Origin")
        self.ax.legend(loc="upper right")

    def update(self, frameId: int):
        x, y, z, velocity, relTime = self.dataLoader.getFrameData(frameId)
        isMoving = velocity.abs() >= self.velocityThreshold
        isStatic = velocity.abs() < self.velocityThreshold

        if self.removeStaticPoints:
            outX, outY, outZ, outV = x[isMoving].tolist(), y[isMoving].tolist(), z[isMoving].tolist(), velocity[isMoving].tolist()
        elif self.persistStaticPoints:
            self.staticX.extend(x[isStatic].tolist())
            self.staticY.extend(y[isStatic].tolist())
            self.staticZ.extend(z[isStatic].tolist())
            self.staticV.extend(velocity[isStatic].tolist())
            outX = self.staticX + x[isMoving].tolist()
            outY = self.staticY + y[isMoving].tolist()
            outZ = self.staticZ + z[isMoving].tolist()
            outV = self.staticV + velocity[isMoving].tolist()
        else:
            outX, outY, outZ, outV = x.tolist(), y.tolist(), z.tolist(), velocity.tolist()

        self.scatter._offsets3d = (outX, outY, outZ)
        self.scatter.set_array(np.array(outV))
        
        self.timeText.set_text(f"Time: {relTime:.2f}s / {self.dataLoader.totalDuration:.2f}s")
        return self.scatter, self.timeText

    def start(self) -> FuncAnimation:
        self.initializeAxes()
        self.animation = FuncAnimation(
            self.fig,
            self.update,
            frames=self.dataLoader.uniqueFrames,
            interval=self.animationInterval,
            blit=False,
            repeat=self.repeatAnimation
        )
        return self.animation

    def save(self, outputPath: str) -> None:
        if self.animation is None:
            self.start()
        
        writer = FFMpegWriter(fps=self.fps, metadata=dict(artist="RadarVisualizer"), bitrate=BITRATE)
        self.animation.save(outputPath, writer=writer)
        plt.close(self.fig)

loader = RadarDataLoader(FILE_PATH)
if loader.load():
    visualizer = RadarVisualizer(
        loader, 
        fps=FPS, 
        velocityThreshold=VELOCITY_THRESHOLD,
        removeStaticPoints=REMOVE_STATIC,
        persistStaticPoints=PERSIST_STATIC
    )
    visualizer.save(OUTPUT_VIDEO_PATH)
    display(Video(OUTPUT_VIDEO_PATH, embed=True))